# SCRIPTY API (Strict Generation Mode)
This notebook runs the finalized, strict story generation API.

### Step 1: Environment & Autoreload
Run this to ensure the kernel picks up all backend script changes immediately.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
!{sys.executable} -m pip install fastapi uvicorn nest_asyncio requests

### Step 2: Clear Memory & Start Server
Running this cell will re-initialize the Story Engine and start a clean FastAPI server in the background.
**NOTE**: This cell will automatically try to kill any existing server on port 8000.

In [2]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import threading
import os
import sys
import importlib
import subprocess
import time

nest_asyncio.apply()
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# --- 1. CLEANUP OLD PROCESSES ---
try:
    # Kill any process running on port 8000
    subprocess.run("lsof -t -i:8000 | xargs kill -9", shell=True, stderr=subprocess.DEVNULL)
    print("Cleaning up existing port 8000 processes...")
    time.sleep(1)
except:
    pass

# --- 2. FORCE RELOAD ---
import control_system
import location_engine
import story_engine
importlib.reload(control_system)
importlib.reload(location_engine)
importlib.reload(story_engine)

from story_engine import StoryEngine
from location_engine import get_location_context

# --- 3. DEFINE APP ---
app = FastAPI(title="SCRIPTY - TOTAL ELIMINATION MODE")

class StoryRequest(BaseModel):
    genre: str
    theme: str
    location: str
    year: int

class StoryResponse(BaseModel):
    story: str

engine = StoryEngine(data_dir="data_processed")

@app.post("/generate-story", response_model=StoryResponse)
async def generate_story(req: StoryRequest):
    try: 
        # The engine handles state, act transitions, and strict safety validation
        story_text = engine.create_structured_story(
            req.genre, 
            req.theme, 
            req.location, 
            req.year
        )
        return StoryResponse(story=story_text)
    except Exception as e:
        print(f"Generation Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"status": "safe", "message": "SCRIPTY Strict Generation Server Active"}

def start_server():
    try:
        config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
        server = uvicorn.Server(config)
        server.run()
    except:
        pass

# --- 4. START SERVER ---
thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("STRICT Server is running on http://localhost:8000")

Cleaning up existing port 8000 processes...
STRICT Server is running on http://localhost:8000


### Step 3: Test Output (STRICT VERIFICATION)
Run this to verify that NO dataset text is leaking into the story.

In [3]:
import requests
import time

time.sleep(1)

test_payload = {
    "genre": "Historical Fiction",
    "theme": "Heritage",
    "location": "Hyderabad",
    "year": 1920
}

try:
    response = requests.post("http://localhost:8000/generate-story", json=test_payload)

    print("Status Code:", response.status_code)

    # 🔍 Always print raw response first
    print("Raw Response:", response.text)

    if response.status_code == 200:
        try:
            data = response.json()   # ✅ safe parsing
            print("STRICT STORY OUTPUT:")
            print("="*30)
            print(data.get("story", "No story field found"))
            print("="*30)
        except Exception as e:
            print("❌ JSON parsing failed:", e)

    else:
        try:
            error_data = response.json()
            print(f"API Error {response.status_code}: {error_data.get('detail', 'Unknown error')}")
        except:
            print(f"API Error {response.status_code}: {response.text}")

except Exception as e:
    print(f"Connection error: {e}")

Status Code: 200
Raw Response: {"story":"Introduction:\nThe shadow of the British Residency stretched across Hyderabad in 1920. Priya, a merchant, watched the crowds passing by.\n\nConflict:\nA deep Heritage emerged when Priya discovered stolen Nizam treasury bonds. The Local Freedom Fighters were already watching.\n\nClimax:\nWith the stolen Nizam treasury bonds held firmly in hand, Priya stood their ground against Vikram in a final confrontation.\n\nResolution:\nIn the end, Priya ensured that the stolen Nizam treasury bonds was safe. Hyderabad returned to its usual rhythm, though changed."}
STRICT STORY OUTPUT:
Introduction:
The shadow of the British Residency stretched across Hyderabad in 1920. Priya, a merchant, watched the crowds passing by.

Conflict:
A deep Heritage emerged when Priya discovered stolen Nizam treasury bonds. The Local Freedom Fighters were already watching.

Climax:
With the stolen Nizam treasury bonds held firmly in hand, Priya stood their ground against Vikram 

In [9]:
import requests
import time

time.sleep(1)

test_payload = {
    "genre": "Historical Fiction",
    "theme": "Heritage",
    "location": "Varanasi",
    "year": 2020
}

response = requests.post(
    "http://127.0.0.1:8000/generate-story",
    json=test_payload,
    proxies={"http": None, "https": None}
)

print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 200
Response: {"story":"Introduction:\nIshaan, a activist honoring their family legacy, stood before the the town center in Varanasi. It was 2020, a period defined by neutral.\n\nConflict:\nThe peace was shattered by a missing family heirloom, drawing Ishaan into a conflict involving the traveling merchants.\n\nClimax:\nWith the a missing family heirloom held firmly in hand, Ishaan stood their ground against Sameer in a final confrontation.\n\nResolution:\nAs the dust settled, Ishaan realized their purpose as a activist was fulfilled. The a missing family heirloom was no longer a threat."}


In [10]:
for i in range(3):
    response = requests.post("http://localhost:8000/generate-story", json=test_payload)
    print(f"\n--- RUN {i+1} ---\n")
    print(response.json()["story"])


--- RUN 1 ---

Introduction:
In the heart of Varanasi, near the local square, Meera was contemplating their background of competing in the global market.

Conflict:
A deep Heritage emerged when Meera discovered a missing family heirloom. The local villagers were already watching.

Climax:
The tension between the local villagers and Meera exploded during the struggle for the a missing family heirloom.

Resolution:
The mystery of the a missing family heirloom was resolved, and Meera could finally rest. Varanasi would never be the same.

--- RUN 2 ---

Introduction:
In the heart of Varanasi, near the local square, Meera was contemplating their background of competing in the global market.

Conflict:
A deep Heritage emerged when Meera discovered a missing family heirloom. The local villagers were already watching.

Climax:
The situation reached its peak. Meera confronted Ishaan over the a missing family heirloom, while members of the local villagers closed in.

Resolution:
The mystery of 